# Compare Saved Observation With Three Models

这个 notebook 参考 `deploy_debug/inspect_saved_single_maskaware_action_step0.ipynb`，但不修改原文件。

用途：
- 读取同一份保存下来的单臂观测；
- 分别送给 `original` / `mask_aware` / `tcp_anchor` 三个模型重推理；
- 用 **3 个独立 cell** 分开显示三条预测轨迹，不画在同一张图里。


In [ ]:
from __future__ import annotations

import json
import os
import sys
from copy import deepcopy
from pathlib import Path

import cv2
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import plotly.io as pio
import torch
import yaml
import torchvision
from easydict import EasyDict as edict

WORKSPACE_ROOT = Path('/home/haoxiang/rise2_mask_aware')
for p in [WORKSPACE_ROOT, WORKSPACE_ROOT / 'airexo', WORKSPACE_ROOT / 'easyrobot']:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

os.chdir(WORKSPACE_ROOT)
print('cwd =', os.getcwd())
from policy import RISE2
from dataset.projector import SingleArmProjector
from dataset.data_utils import ImageProcessor, resize_image
from utils.training import set_seed

pio.renderers.default = 'notebook_connected'
torch.cuda.set_device(0)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device =', device)


In [ ]:
# 这里选择同一份观测。
# 可以替换成 deploy_capture_original / deploy_capture_single_mask_aware / deploy_capture_single_tcp_anchor 中任意一份。
# CAPTURE_ROOT = Path('/data/haoxiang/data/deploy_debug_260419/deploy_capture_single_mask_aware/capture_20260419_100346')
# STEP_ID = 40
# STEP_STEM = f'step_{STEP_ID:06d}'

CAPTURE_ROOT = Path('/data/haoxiang/data/deploy_debug_260419/deploy_capture_single_mask_aware/capture_20260419_100346')
STEP_ID = 80

STEP_STEM = f'step_{STEP_ID:06d}'
CALIB_RISE2_PATH = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/calib/rise2_calib_single_foar_purplebox.npy')

META_PATH = CAPTURE_ROOT / 'meta.json'
RGB_PATH = CAPTURE_ROOT / 'rgb' / f'{STEP_STEM}.png'
DEPTH_PATH = CAPTURE_ROOT / 'depth' / f'{STEP_STEM}.png'
MASK_PATH = CAPTURE_ROOT / 'mask' / f'{STEP_STEM}.png'
ACTION_PATH = CAPTURE_ROOT / 'actions' / f'{STEP_STEM}.npy'
PROPRIO_PATH = CAPTURE_ROOT / 'proprio' / f'{STEP_STEM}.npy'
JOINT_PATH = CAPTURE_ROOT / 'joint' / f'{STEP_STEM}.npy'
TCP_CAMERA_PATH = CAPTURE_ROOT / 'tcp_camera' / f'{STEP_STEM}.npy'
ANCHOR_POINTS_PATH = CAPTURE_ROOT / 'anchor_points' / f'{STEP_STEM}.npy'
POINT_MAX = 120000

meta = json.loads(META_PATH.read_text(encoding='utf-8'))
rgb = cv2.cvtColor(cv2.imread(str(RGB_PATH), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
depth = cv2.imread(str(DEPTH_PATH), cv2.IMREAD_UNCHANGED)
saved_action = np.load(ACTION_PATH)
proprio = np.load(PROPRIO_PATH)
joint = np.load(JOINT_PATH)
mask = cv2.imread(str(MASK_PATH), cv2.IMREAD_UNCHANGED) if MASK_PATH.exists() else None
saved_tcp_camera = np.load(TCP_CAMERA_PATH) if TCP_CAMERA_PATH.exists() else None
saved_anchor_points = np.load(ANCHOR_POINTS_PATH) if ANCHOR_POINTS_PATH.exists() else None
print(meta)
print('saved_action shape', saved_action.shape)
print('proprio shape', proprio.shape)
print('joint shape', joint.shape)
print('mask exists', mask is not None, 'tcp_camera exists', saved_tcp_camera is not None, 'anchor_points exists', saved_anchor_points is not None)


In [ ]:
def create_point_cloud(colors, depths, cam_intrinsics, config, depth_scale=1000.0, rescale_factor=1.0):
    h, w = depths.shape
    fx, fy = cam_intrinsics[0, 0] * rescale_factor, cam_intrinsics[1, 1] * rescale_factor
    cx, cy = cam_intrinsics[0, 2] * rescale_factor, cam_intrinsics[1, 2] * rescale_factor
    color_o3d = o3d.geometry.Image(colors.astype(np.uint8))
    depth_o3d = o3d.geometry.Image(depths.astype(np.float32))
    camera_intrinsics = o3d.camera.PinholeCameraIntrinsic(width=w, height=h, fx=fx, fy=fy, cx=cx, cy=cy)
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(color_o3d, depth_o3d, depth_scale, convert_rgb_to_intensity=False)
    cloud = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, camera_intrinsics)
    bbox3d = o3d.geometry.AxisAlignedBoundingBox(config.deploy.workspace.min, config.deploy.workspace.max)
    cloud = cloud.crop(bbox3d)
    cloud = cloud.voxel_down_sample(config.data.voxel_size)
    return cloud

def create_input(colors, depths, cam_intrinsics, config, depth_scale=1000.0, rescale_factor=1.0):
    cloud = create_point_cloud(colors, depths, cam_intrinsics, config, depth_scale=depth_scale, rescale_factor=rescale_factor)
    points = np.asarray(cloud.points)
    coords = np.ascontiguousarray(points / config.data.voxel_size, dtype=np.int32)
    return coords, points, cloud

def create_batch(coords, points):
    import MinkowskiEngine as ME
    coords_batch, feats_batch = ME.utils.sparse_collate([coords], [points.astype(np.float32)])
    return coords_batch, feats_batch

def process_state(state, config, to_control=True):
    state = np.asarray(state).copy()
    if to_control:
        state[..., 0:3] = (state[..., 0:3] + 1) / 2.0 * (config.data.normalization.trans_max - config.data.normalization.trans_min) + config.data.normalization.trans_min
        state[..., 9] = (state[..., 9] + 1) / 2.0 * config.data.normalization.max_gripper_width
    else:
        state[..., 0:3] = (state[..., 0:3] - config.data.normalization.trans_min) / (config.data.normalization.trans_max - config.data.normalization.trans_min) * 2.0 - 1
        state[..., 9] = state[..., 9] / config.data.normalization.max_gripper_width * 2.0 - 1
    return state

def build_mask_aware_cfg(config):
    default_json_mask_cfg = {'single_json': None, 'cam_base_mode': 'base_to_cam__predef', 'use_agent_intrinsics': True, 'intrinsics_npy': None, 'intrinsic_selector': 'first', 'single_urdf': str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/zihao_single_worldbase_y_neg90/left_robot_grav.urdf').resolve()), 'width': 1280, 'height': 720, 'near_plane': 0.01, 'far_plane': 100.0}
    default_cfg = {'enabled': False, 'enable_3d_filter': True, 'enable_2d_reweight': True, 'mask_threshold': 0, 'mask_white_is_untrusted': True, 'dilate_radius': 10, 'debug_save_mask': False, 'debug_save_every': 10, 'debug_dir': 'mask_debug_real', 'debug_print_stats': True, 'json_mask': default_json_mask_cfg}
    raw_cfg = getattr(config, 'mask_aware', {})
    raw_cfg = dict(raw_cfg) if raw_cfg is not None else {}
    merged = deepcopy(default_cfg)
    for k, v in raw_cfg.items():
        if v is None or k not in merged:
            continue
        if k == 'json_mask' and isinstance(v, dict):
            json_cfg = deepcopy(default_json_mask_cfg)
            json_cfg.update({kk: vv for kk, vv in v.items() if vv is not None})
            merged[k] = json_cfg
        else:
            merged[k] = v
    merged['json_mask'] = edict(merged['json_mask'])
    return edict(merged)

def build_tcp_anchor_cfg(config):
    default_cfg = {'enabled': False, 'anchor_num_points': 7, 'anchor_radius_scale': 0.75, 'patch_weight_mode': 'anchor_only', 'tcp_patch_radius': 1, 'tcp_patch_weight_floor': 1.0, 'mask_dilate_kernel': 9}
    raw_cfg = getattr(config, 'tcp_anchor', {})
    raw_cfg = dict(raw_cfg) if raw_cfg is not None else {}
    merged = deepcopy(default_cfg)
    for k, v in raw_cfg.items():
        if v is not None and k in merged:
            merged[k] = v
    merged['enabled'] = bool(merged['enabled'])
    merged['anchor_num_points'] = int(merged['anchor_num_points'])
    merged['anchor_radius_scale'] = float(merged['anchor_radius_scale'])
    return edict(merged)

def build_image_mask_weight(mask01, image_processor):
    mask_tensor = torch.from_numpy(mask01[np.newaxis].astype(np.float32))
    mask_tensor = resize_image(mask_tensor, image_processor.img_size, interpolation=torchvision.transforms.InterpolationMode.NEAREST)
    mask_ratio = image_processor.image_coord_pooling(mask_tensor)
    image_mask_weight = (1.0 - mask_ratio).clamp(0.0, 1.0).to(torch.float32)
    return image_mask_weight

TCP_ANCHOR_OFFSETS_7 = np.asarray([[0.,0.,0.],[1.,0.,0.],[-1.,0.,0.],[0.,1.,0.],[0.,-1.,0.],[0.,0.,1.],[0.,0.,-1.]], dtype=np.float32)
def build_tcp_anchor_points(tcp_camera, voxel_size, radius_scale):
    radius = float(voxel_size * radius_scale)
    offsets = TCP_ANCHOR_OFFSETS_7 * radius
    return offsets + np.asarray(tcp_camera[:3], dtype=np.float32)[None, :]

def create_input_with_anchor(colors, depths, cam_intrinsics, config, tcp_camera, depth_scale=1000.0, rescale_factor=1.0):
    cloud = create_point_cloud(colors, depths, cam_intrinsics, config, depth_scale=depth_scale, rescale_factor=rescale_factor)
    points = np.asarray(cloud.points)
    if tcp_camera is not None and config.tcp_anchor.enabled:
        anchor_points = build_tcp_anchor_points(tcp_camera, voxel_size=config.data.voxel_size, radius_scale=config.tcp_anchor.anchor_radius_scale).astype(np.float32)
        points = np.concatenate([points, anchor_points], axis=0)
    coords = np.ascontiguousarray(points / config.data.voxel_size, dtype=np.int32)
    return coords, points, cloud

def project_point_to_patch_coord(point_xyz, intrinsics, orig_shape, img_size, img_coord_size):
    z = float(point_xyz[2])
    if z <= 1e-6:
        return None
    fx, fy = float(intrinsics[0,0]), float(intrinsics[1,1])
    cx, cy = float(intrinsics[0,2]), float(intrinsics[1,2])
    u = fx * float(point_xyz[0]) / z + cx
    v = fy * float(point_xyz[1]) / z + cy
    h0, w0 = int(orig_shape[0]), int(orig_shape[1])
    if u < 0 or u >= w0 or v < 0 or v >= h0:
        return None
    h1, w1 = int(img_size[0]), int(img_size[1])
    ph, pw = int(img_coord_size[0]), int(img_coord_size[1])
    u_resized = u * (w1 / max(w0, 1))
    v_resized = v * (h1 / max(h0, 1))
    patch_j = int(np.clip(np.floor(u_resized * pw / max(w1, 1)), 0, pw - 1))
    patch_i = int(np.clip(np.floor(v_resized * ph / max(h1, 1)), 0, ph - 1))
    return patch_i, patch_j

def restore_tcp_patch_weight(image_mask_weight, tcp_camera, intrinsics, orig_shape, image_processor, tcp_anchor_cfg):
    patch_coord = project_point_to_patch_coord(tcp_camera[:3], intrinsics, orig_shape, image_processor.img_size, image_processor.image_coord_pooling.output_size)
    if patch_coord is None:
        return image_mask_weight
    patch_i, patch_j = patch_coord
    h, w = image_mask_weight.shape[-2], image_mask_weight.shape[-1]
    for i in range(max(0, patch_i - tcp_anchor_cfg.tcp_patch_radius), min(h, patch_i + tcp_anchor_cfg.tcp_patch_radius + 1)):
        for j in range(max(0, patch_j - tcp_anchor_cfg.tcp_patch_radius), min(w, patch_j + tcp_anchor_cfg.tcp_patch_radius + 1)):
            image_mask_weight[0, i, j] = max(float(image_mask_weight[0, i, j]), tcp_anchor_cfg.tcp_patch_weight_floor)
    return image_mask_weight

def add_tcp_marker(fig, tcp_pose, color, radius=0.006, size=2.2, opacity=0.95):
    center = np.asarray(tcp_pose[:3], dtype=np.float64)
    offsets = np.array([[0.0,0.0,0.0],[radius,0.0,0.0],[-radius,0.0,0.0],[0.0,radius,0.0],[0.0,-radius,0.0],[0.0,0.0,radius],[0.0,0.0,-radius]])
    pts = center[None, :] + offsets
    fig.add_trace(go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers', marker=dict(size=size, color=color, opacity=opacity), showlegend=False))

def run_model(model_name, config_path, ckpt_path, use_mask, use_tcp_anchor):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = edict(yaml.load(f, Loader=yaml.FullLoader))
    config.data.normalization.trans_min = np.asarray(config.data.normalization.trans_min)
    config.data.normalization.trans_max = np.asarray(config.data.normalization.trans_max)
    config.mask_aware = build_mask_aware_cfg(config)
    config.tcp_anchor = build_tcp_anchor_cfg(config)
    if not use_mask:
        config.mask_aware.enabled = False
    if not use_tcp_anchor:
        config.tcp_anchor.enabled = False
    set_seed(config.deploy.seed)
    projector = SingleArmProjector(str(CALIB_RISE2_PATH), meta['camera_serial'])
    policy = RISE2(num_action=config.data.num_action, obs_feature_dim=config.model.obs_feature_dim, cloud_enc_dim=config.model.cloud_enc_dim, image_enc_dim=config.model.image_enc_dim, action_dim=10, hidden_dim=config.model.hidden_dim, nheads=config.model.nheads, num_attn_layers=config.model.num_attn_layers, dim_feedforward=config.model.dim_feedforward, dropout=config.model.dropout, image_enc=config.model.image_enc, interp_fn_mode=config.model.interp_fn_mode, image_enc_finetune=config.model.image_enc_finetune, image_enc_dtype=config.model.image_enc_dtype).to(device)
    policy.load_state_dict(torch.load(ckpt_path, map_location=device), strict=False)
    policy.eval()
    image_enc = config.model.image_enc
    if image_enc == 'resnet18':
        img_size = config.data.aligner.img_size_resnet
        img_coord_size = config.data.aligner.img_coord_size_resnet
    elif image_enc.startswith('dinov2'):
        img_size = config.data.aligner.img_size_dinov2
        img_coord_size = config.data.aligner.img_coord_size_dinov2
    elif image_enc.startswith('dinov3'):
        img_size = config.data.aligner.img_size_dinov3
        img_coord_size = config.data.aligner.img_coord_size_dinov3
    else:
        raise ValueError(image_enc)
    image_processor = ImageProcessor(img_size=img_size, img_coord_size=img_coord_size, voxel_size=config.data.voxel_size, img_mean=config.data.normalization.img_mean, img_std=config.data.normalization.img_std)
    intr_data = np.load(CALIB_RISE2_PATH, allow_pickle=True)
    if isinstance(intr_data, np.ndarray) and intr_data.shape == ():
        intr_data = intr_data.item()
    intrinsic = np.asarray(intr_data['intrinsics'][meta['camera_serial']], dtype=np.float32)
    depths_for_cloud = depth.copy()
    if use_mask and mask is not None:
        depths_for_cloud[mask > 127] = 0
    if use_tcp_anchor:
        coords, points, cloud = create_input_with_anchor(rgb, depths_for_cloud, cam_intrinsics=intrinsic, config=config, tcp_camera=saved_tcp_camera, depth_scale=1000.0, rescale_factor=1.0)
    else:
        coords, points, cloud = create_input(rgb, depths_for_cloud, cam_intrinsics=intrinsic, config=config, depth_scale=1000.0, rescale_factor=1.0)
    image_coords = image_processor.get_image_coordinates(depth, intrinsic, 1000.0)
    colors_t, image_coords_t = image_processor.preprocess_images(rgb, image_coords)
    image_mask_weight = None
    if use_mask and mask is not None:
        image_mask_weight = build_image_mask_weight((mask > 127).astype(np.float32), image_processor)
        if use_tcp_anchor and saved_tcp_camera is not None:
            image_mask_weight = restore_tcp_patch_weight(image_mask_weight, saved_tcp_camera, intrinsic, depth.shape[:2], image_processor, config.tcp_anchor)
    import MinkowskiEngine as ME
    coords_batch, feats_batch = create_batch(coords, points)
    cloud_data = ME.SparseTensor(feats_batch.to(device), coords_batch.to(device))
    colors_model = colors_t.unsqueeze(0).to(device)
    image_coords_model = image_coords_t.unsqueeze(0).to(device)
    if image_mask_weight is not None:
        image_mask_weight = image_mask_weight.unsqueeze(0).to(device)
    with torch.inference_mode():
        pred_raw_action = policy(cloud_data, colors_model, image_coords_model, image_mask_weight=image_mask_weight, actions=None).squeeze(0).cpu().numpy()
    pred_action_camera = process_state(pred_raw_action.copy(), config, to_control=True)
    current_tcp_camera = projector.project_tcp_to_camera_coord(proprio[:9], rotation_rep='rotation_6d')
    saved_action_camera = projector.project_tcp_to_camera_coord(saved_action[:9], rotation_rep='rotation_6d')
    return {'name': model_name, 'config': config, 'cloud': cloud, 'pred_action_camera': pred_action_camera, 'current_tcp_camera': current_tcp_camera, 'saved_action_camera': saved_action_camera}

def render_result(result, extra_anchor_points=None):
    cloud = result['cloud']
    pc_points = np.asarray(cloud.points)
    pc_colors = np.asarray(cloud.colors)
    if pc_points.shape[0] > POINT_MAX:
        idx = np.linspace(0, pc_points.shape[0] - 1, POINT_MAX).astype(np.int64)
        pc_points = pc_points[idx]
        pc_colors = pc_colors[idx]
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=pc_points[:,0], y=pc_points[:,1], z=pc_points[:,2], mode='markers', marker=dict(size=1.5, color=pc_colors, opacity=0.55), name=f"{result['name']}_point_cloud"))
    if extra_anchor_points is not None:
        anchors = np.asarray(extra_anchor_points, dtype=np.float64)
        fig.add_trace(go.Scatter3d(x=anchors[:,0], y=anchors[:,1], z=anchors[:,2], mode='markers', marker=dict(size=2.5, color='royalblue', opacity=0.9), name='saved_anchor_points'))
    add_tcp_marker(fig, result['current_tcp_camera'], 'deepskyblue', radius=0.007, size=2.8, opacity=1.0)
    for raw_tcp in result['pred_action_camera']:
        add_tcp_marker(fig, raw_tcp, 'dimgray', radius=0.006, size=2.0, opacity=0.9)
    add_tcp_marker(fig, result['saved_action_camera'], 'dimgray', radius=0.006, size=2.4, opacity=1.0)
    fig.update_layout(title=f"{result['name']} | same observation replay | step {STEP_ID}", scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='data'), width=1250, height=920, showlegend=extra_anchor_points is not None)
    return fig


In [ ]:
ORIGINAL_RESULT = run_model(
    model_name='original',
    config_path=WORKSPACE_ROOT / 'configs/single_foar_purplebox_original.yaml',
    ckpt_path=Path('/data/haoxiang/logs/single_foar_purplebox_original/policy_last.ckpt'),
    use_mask=False,
    use_tcp_anchor=False,
)
print('original horizon shape =', ORIGINAL_RESULT['pred_action_camera'].shape)


In [ ]:
MASK_AWARE_RESULT = run_model(
    model_name='mask_aware',
    config_path=WORKSPACE_ROOT / 'configs/single_foar_purplebox_eval_json.yaml',
    ckpt_path=Path('/data/haoxiang/logs/single_foar_purplebox_mask_aware/policy_last.ckpt'),
    use_mask=True,
    use_tcp_anchor=False,
)
print('mask_aware horizon shape =', MASK_AWARE_RESULT['pred_action_camera'].shape)


In [ ]:
TCP_ANCHOR_RESULT = run_model(
    model_name='tcp_anchor',
    config_path=WORKSPACE_ROOT / 'configs/single_foar_purplebox_eval_tcp_anchor.yaml',
    ckpt_path=Path('/data/haoxiang/logs/single_foar_purplebox_tcp_anchor/policy_last.ckpt'),
    use_mask=True,
    use_tcp_anchor=True,
)
print('tcp_anchor horizon shape =', TCP_ANCHOR_RESULT['pred_action_camera'].shape)


In [ ]:
render_result(ORIGINAL_RESULT)


In [ ]:
render_result(MASK_AWARE_RESULT)


In [ ]:
render_result(TCP_ANCHOR_RESULT, extra_anchor_points=saved_anchor_points)
